# Document Pre-processing for Knowledge Tuning

## Overview

This notebook demonstrates a complete document preprocessing pipeline designed specifically for **knowledge tuning** with sdg-hub. 

## What This Notebook Does

This preprocessing pipeline transforms raw documents (PDFs, Word docs, etc.) into seed data for data generation:

1. **Document Parsing**: Converts raw documents to structured markdown format
2. **Chunking**: Splits documents into manageable chunks while preserving structure and context
3. **Seed Data Creation**: Formats chunks with in-context learning (ICL) templates for effective knowledge tuning

## Prerequisites

- We will use the existing InstructLab document parser (`docparser_v2.py`) and Document parsing configuration (`docling_v2_config.yaml`)
- Raw pdf documents in the `document_collection/` directory


In [6]:
!uv remove logger-config
!uv add loguru


Resolved 274 packages in 100ms                                       
Uninstalled 2 packages in 0.95ms
 - logger-config==0.3
 - typer-slim==0.24.0
Resolved 276 packages in 116ms                                       
Prepared 1 package in 5.02s                                              
Installed 1 package in 92ms                                 
 + loguru==0.7.3


In [2]:
# Step 1: Document Processing Pipeline
# Define the directory containing raw documents to be processed
input_data_dir = "data/research_papers/"
output_data_dir = "data/research_papers_md/"

In [ ]:
# Run the document parser to convert documents to markdown
# - input-dir: Directory containing source documents
# - output-dir: Directory where processed markdown files will be saved
# - c: Configuration file specifying parsing parameters
!CUDA_VISIBLE_DEVICES=1,2,3,4,5,6,7 python scripts/docparser_v2.py --input-dir {input_data_dir} --output-dir {output_data_dir} -c scripts/docling_v2_config.yaml

In [11]:
# Step 3: Load Processed Document
import glob

# In our example above docling step produces markdown of all the pdf files in the document_collection
all_text = []
for file in glob.glob(f"{output_data_dir}/*.md"):
    with open(file, "r") as f:
        all_text.append(f.read())

all_text[0][:200]

'## Router-R1: Teaching LLMs Multi-Round Routing and Aggregation via Reinforcement Learning\n\n## Haozhen Zhang 1 ∗ Tao Feng Jiaxuan You\n\nUniversity of Illinois at Urbana-Champaign 1\n\n{haozhenz,taofeng2,'

In [12]:
# Step 4: Text Chunking and Dataset Creation

from markdown_it import MarkdownIt
from typing import List
import datasets


def chunk_markdown(text: str, max_tokens: int = 200, overlap: int = 50) -> List[str]:
    """
    Splits Markdown text into chunks at block-level elements
    (headings, paragraphs, lists, tables, code, blockquotes).
    Adds overlap (in words) between all consecutive chunks.

    Args:
        text: The markdown text to be chunked
        max_tokens: Maximum number of words per chunk
        overlap: Number of overlapping words between consecutive chunks

    Returns:
        List of text chunks with specified overlap
    """

    # Initialize markdown parser to understand document structure
    md = MarkdownIt()
    tokens = md.parse(text)

    # Group tokens into block-level segments to preserve markdown structure
    # This ensures we don't split in the middle of headings, lists, etc.
    blocks = []
    buf = []
    for tok in tokens:
        if tok.block and tok.type.endswith("_open"):
            buf = []
        elif tok.block and tok.type.endswith("_close"):
            if buf:
                blocks.append("\n".join(buf).strip())
                buf = []
        elif tok.content:
            buf.append(tok.content)
    if buf:
        blocks.append("\n".join(buf).strip())

    # Split blocks into chunks with overlap to maintain context continuity
    chunks = []
    current_words = []
    for block in blocks:
        words = block.split()
        for w in words:
            current_words.append(w)
            if len(current_words) >= max_tokens:
                # Emit a complete chunk
                chunks.append(" ".join(current_words))
                # Prepare next buffer with overlap from the end of this chunk
                # This ensures context continuity between chunks
                current_words = current_words[-overlap:] if overlap > 0 else []

    # Add any remaining words as the final chunk
    if current_words:
        chunks.append(" ".join(current_words))

    return chunks


chunks = []
for text in all_text:
    chunks.extend(chunk_markdown(text, max_tokens=5000, overlap=1000))


# Prepare seed data for the SDG-Hub knowledge pipeline.
#
# The seed data requires the following fields:
#   - document_outline: A concise title or summary that accurately represents the entire document.
#     For documents covering multiple themes, consider providing multiple outlines (one per section).
#   - icl_document: A representative sample extract from the document. This may include tables, code snippets, definitions, etc.
#   - icl_query_1, icl_query_2, icl_query_3: Three questions based on the icl_document sample.
#   - domain: The domain or subject area of the document.
#
# The code below creates a HuggingFace Dataset from the document chunks,
# then maps the required ICL fields to each entry, and finally saves the result as a JSONL file.

seed_data = datasets.Dataset.from_dict({"document": chunks})

icl = {
    "document_outline": "The document contains excerpts from FINTRAC regulations designed to combat money laundering and terrorist financing in Canada",
    "icl_document": "## Overview\n\nThis guidance came into effect on June 1, 2021.\n\n\nThis guidance explains the methods that can be used by reporting entities\n(REs) to verify the identity of a person or an entity.\n\n\n## 1. Meaning of verifying the identity of a person or an entity\n\nIt means to use the methods described in this guidance to ensure that the\ninformation in an identification document or from other informational\nsources matches the information that the person or entity provided.\n\n\nVerifying identity is a foundational element of Canada's anti-money\nlaundering and anti-terrorist financing regime and a key component of an\nRE's relationship with clients. It helps you to know your clients and to\nunderstand and assess any risk that may be associated to their\ntransactions or activities.\n\n\n## 2. How to verify the identity of a person\n\nYou can use any of the 5 methods described below to identify a person:\n\n- 2.1 Government-issued photo identification method\n\n- 2.2 Credit file method\n\n- 2.3 Dual-process method\n\n- 2.4 Affiliate or member method\n\n- 2.5 Reliance method\n",
    "icl_query_1": "In Canada, what are the methods for verifying someone's identity?",
    "icl_query_2": "In Canada, why is it important to confirm a client's identity?",
    "icl_query_3": "In Canada, can I use Reliance method to verify identity of a person?",
    "domain": "Finance",
}

# Map the ICL fields to each document chunk (if you want to use the same ICL for all, as shown here)
seed_data = seed_data.map(lambda x: icl)

# Save the seed data to a JSONL file for downstream use
seed_data.to_json("data/seed_data.jsonl", orient="records", lines=True)

Map:   0%|          | 0/246 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

7493383

### Next Steps:
- The seed_data.jsonl file is now ready for the knowledge tuning pipeline.
- You can now refer to the [knowledge generation](knowledge_generation.ipynb) notebook

In [13]:
seed_data

Dataset({
    features: ['document', 'document_outline', 'icl_document', 'icl_query_1', 'icl_query_2', 'icl_query_3', 'domain'],
    num_rows: 246
})

In [14]:
print(seed_data[0]['document'])

Router-R1: Teaching LLMs Multi-Round Routing and Aggregation via Reinforcement Learning Haozhen Zhang 1 ∗ Tao Feng Jiaxuan You University of Illinois at Urbana-Champaign 1 {haozhenz,taofeng2,jiaxuan}@illinois.edu, wazhz14@gmail.com Abstract The rapid emergence of diverse large language models (LLMs) has spurred the development of LLM routers that assign user queries to the most suitable model. However, existing LLM routers typically perform a single-round, one-to-one mapping ( i.e. , assigning each query to a single model in isolation), which limits their capability to tackle complex tasks that demand the complementary strengths of multiple LLMs. In this paper, we present Router-R1 , a reinforcement learning (RL)-based framework that formulates multi-LLM routing and aggregation as a sequential decision process. Router-R1 instantiates the router itself as a capable LLM, leveraging its reasoning ability to interleave 'think' actions (internal deliberation) with 'route' actions (dynamic m

In [9]:
import textwrap
wprint = lambda x: print(textwrap.fill(x, width=100))
wprint(seed_data[0]['document'])


Router-R1: Teaching LLMs Multi-Round Routing and Aggregation via Reinforcement Learning Haozhen
Zhang 1 ∗ Tao Feng Jiaxuan You University of Illinois at Urbana-Champaign 1
{haozhenz,taofeng2,jiaxuan}@illinois.edu, wazhz14@gmail.com Abstract The rapid emergence of diverse
large language models (LLMs) has spurred the development of LLM routers that assign user queries to
the most suitable model. However, existing LLM routers typically perform a single-round, one-to-one
mapping ( i.e. , assigning each query to a single model in isolation), which limits their capability
to tackle complex tasks that demand the complementary strengths of multiple LLMs. In this paper, we
present Router-R1 , a reinforcement learning (RL)-based framework that formulates multi-LLM routing
and aggregation as a sequential decision process. Router-R1 instantiates the router itself as a
capable LLM, leveraging its reasoning ability to interleave 'think' actions (internal deliberation)
with 'route' actions (dynamic m

In [10]:
wprint(seed_data[1]['document'])

0.135 | | CoT | 0.126 | 0.358 | 0.160 | 0.168 | 0.208 | 0.046 | 0.224 | 0.184 | | SFT | 0.212 |
0.400 | 0.160 | 0.198 | 0.256 | 0.052 | 0.112 | 0.199 | | RAG | 0.298 | 0.540 | 0.366 | 0.216 |
0.146 | 0.078 | 0.224 | 0.267 | | Search-R1 | 0.328 | 0.510 | 0.324 | 0.236 | 0.278 | 0.090 | 0.272
| 0.291 | | Prompt LLM | 0.300 | 0.580 | 0.340 | 0.268 | 0.262 | 0.108 | 0.448 | 0.329 | | Largest
LLM | 0.296 | 0.578 | 0.354 | 0.278 | 0.274 | 0.104 | 0.480 | 0.338 | | KNN Router | 0.262 | 0.528 |
0.222 | 0.224 | 0.196 | 0.066 | 0.360 | 0.265 | | MLP Router | 0.252 | 0.460 | 0.222 | 0.198 | 0.210
| 0.072 | 0.360 | 0.253 | | BERT Router | 0.230 | 0.516 | 0.192 | 0.216 | 0.206 | 0.058 | 0.312 |
0.247 | | RouterDC | 0.278 | 0.592 | 0.282 | 0.244 | 0.218 | 0.080 | 0.504 | 0.314 | | GraphRouter |
0.276 | 0.586 | 0.280 | 0.234 | 0.180 | 0.076 | 0.448 | 0.297 | | Prompt LLM* | 0.258 | 0.500 |
0.256 | 0.206 | 0.248 | 0.078 | 0.472 | 0.288 | | KNN Router* | 0.236 | 0.478 | 0.232 | 0.154 |
0.234 | 0.072 | 